# Trabajo Práctico Data Warehouse — Siniestros Viales (CABA)

**Tema:** Análisis de siniestralidad vial en la Ciudad de Buenos Aires.

Fuente: [BA Data - Siniestros viales](https://data.buenosaires.gob.ar/dataset/victimas-siniestros-viales)

Este notebook arma el modelo dimensional (1 tabla de hechos + 3 dimensiones) a partir de los dos archivos fuente:
- `hechos.csv` (un registro por siniestro: fecha, hora, ubicación, tipo de vía, vehículos involucrados)
- `victimas.csv` (un registro por persona afectada: sexo, edad, rol, gravedad)

## 1. Carga de datos

Descargá los dos archivos del portal de datos abiertos y subilos a tu Google Drive antes de correr esta celda:
- Hechos (CSV): https://data.buenosaires.gob.ar/dataset/victimas-siniestros-viales/resource/40ec993a-00ad-40e5-936f-0a25f8d2c90b/download
- Víctimas (CSV): https://data.buenosaires.gob.ar/dataset/victimas-siniestros-viales/resource/41beafc0-ca1c-430d-a29d-07c425d0aa20/download

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

hechos_path = "/content/drive/MyDrive/TPDataWarehouseSiniestros/hechos.csv"
victimas_path = "/content/drive/MyDrive/TPDataWarehouseSiniestros/victimas.csv"

# encoding='utf-8-sig' porque los archivos de BA Data traen BOM al inicio de la primera columna
df_hechos = pd.read_csv(hechos_path, encoding='utf-8-sig')
df_victimas = pd.read_csv(victimas_path, encoding='utf-8-sig')

# Normalizamos nombres de columnas por si vienen con espacios o mayúsculas distintas
df_hechos.columns = df_hechos.columns.str.strip().str.lower()
df_victimas.columns = df_victimas.columns.str.strip().str.lower()

print("Columnas de Hechos:", df_hechos.columns.tolist())
print("Columnas de Víctimas:", df_victimas.columns.tolist())
print("Registros Hechos:", len(df_hechos))
print("Registros Víctimas:", len(df_victimas))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/TPDataWarehouseSiniestros/hechos.csv'

## 2. Dimensión Tiempo

Jerarquía: **Año > Mes > Día > Hora** (con el atributo adicional Turno).

El archivo de hechos ya trae `aaaa`, `mm`, `dd` y `hh` como columnas separadas, así que armamos el id numérico directamente sin necesidad de parsear fechas.

In [ ]:
def calcular_turno(hh):
    if pd.isna(hh):
        return 'Sin dato'
    hh = int(hh)
    if 0 <= hh < 6:
        return 'Madrugada'
    elif 6 <= hh < 12:
        return 'Mañana'
    elif 12 <= hh < 19:
        return 'Tarde'
    else:
        return 'Noche'

dim_tiempo = pd.DataFrame()

# id_tiempo con formato AAAAMMDDHH
dim_tiempo['id_tiempo'] = (
    df_hechos['aaaa'].astype(int).astype(str).str.zfill(4)
    + df_hechos['mm'].astype(int).astype(str).str.zfill(2)
    + df_hechos['dd'].astype(int).astype(str).str.zfill(2)
    + df_hechos['hh'].astype(int).astype(str).str.zfill(2)
).astype(int)

dim_tiempo['anio'] = df_hechos['aaaa']
dim_tiempo['mes'] = df_hechos['mm']
dim_tiempo['dia'] = df_hechos['dd']
dim_tiempo['hora'] = df_hechos['hh']
dim_tiempo['turno'] = df_hechos['hh'].apply(calcular_turno)

dim_tiempo = dim_tiempo.drop_duplicates(subset=['id_tiempo']).reset_index(drop=True)

print(dim_tiempo.head())
print("Filas en Dim_Tiempo:", len(dim_tiempo))

## 3. Dimensión Ubicación

Jerarquía: **Comuna > Tipo de calle > Calle**.

A diferencia de Ecobici (donde cada estación tiene un ID fijo en un catálogo), acá no existe un catálogo maestro de ubicaciones: cada siniestro trae su propia comuna/calle como texto libre. Por eso generamos un ID sustituto (surrogate key) a partir de las combinaciones únicas que aparecen en los datos.

In [ ]:
cols_ubicacion = ['comuna', 'tipo_de_calle', 'calle']
dim_ubicacion = df_hechos[cols_ubicacion].drop_duplicates().reset_index(drop=True)
dim_ubicacion['id_ubicacion'] = dim_ubicacion.index + 1
dim_ubicacion = dim_ubicacion[['id_ubicacion', 'comuna', 'tipo_de_calle', 'calle']]

print(dim_ubicacion.head())
print("Filas en Dim_Ubicacion:", len(dim_ubicacion))

## 4. Dimensión Víctima

Jerarquía: **Rango etario > Edad**. Atributos adicionales: Sexo, Rol de la víctima, Tipo de vehículo.

El dataset no tiene un ID persistente de víctima (una misma persona no se puede rastrear entre siniestros), así que modelamos esta dimensión como una **dimensión de perfil deduplicada**: cada combinación única de edad/sexo/rol/vehículo se guarda una sola vez.

In [ ]:
# La edad viene como texto con valores 'SD' (sin dato) -> los convertimos a numérico
df_victimas['edad_numerica'] = pd.to_numeric(df_victimas['edad_victima'], errors='coerce')

bins = [0, 18, 25, 35, 45, 60, 120]
etiquetas = ['Menores de 18', '18-25', '26-35', '36-45', '46-60', 'Mayores de 60']
df_victimas['rango_etario'] = pd.cut(df_victimas['edad_numerica'], bins=bins, labels=etiquetas, right=False)
df_victimas['rango_etario'] = df_victimas['rango_etario'].cat.add_categories(['Sin dato']).fillna('Sin dato')

cols_victima = ['rango_etario', 'sexo_victima', 'rol_victima', 'victima']
dim_victima = df_victimas[cols_victima].drop_duplicates().reset_index(drop=True)
dim_victima['id_victima_perfil'] = dim_victima.index + 1
dim_victima = dim_victima[['id_victima_perfil', 'rango_etario', 'sexo_victima', 'rol_victima', 'victima']]
dim_victima.rename(columns={'victima': 'tipo_vehiculo_victima'}, inplace=True)

print(dim_victima.head())
print("Filas en Dim_Victima:", len(dim_victima))

## 5. Tabla de Hechos: Fact_Siniestros

**Grano:** una fila = una víctima involucrada en un siniestro (join de `victimas` con `hechos` por `id_hecho`).

**Medidas:**
- `cantidad_victimas` (implícita, 1 por registro)
- `es_fallecido` (1 si `gravedad == 'MORTAL'`, 0 en caso contrario)

**Claves foráneas:** `id_tiempo`, `id_ubicacion`, `id_victima_perfil` + el identificador degenerado `id_hecho` (para trazabilidad al siniestro original).

In [ ]:
# Recalculamos las mismas claves en hechos y victimas para poder "engancharlas" con las dimensiones
df_hechos['id_tiempo'] = (
    df_hechos['aaaa'].astype(int).astype(str).str.zfill(4)
    + df_hechos['mm'].astype(int).astype(str).str.zfill(2)
    + df_hechos['dd'].astype(int).astype(str).str.zfill(2)
    + df_hechos['hh'].astype(int).astype(str).str.zfill(2)
).astype(int)

df_hechos = df_hechos.merge(dim_ubicacion, on=cols_ubicacion, how='left')

df_victimas = df_victimas.merge(
    dim_victima.rename(columns={'tipo_vehiculo_victima': 'victima'}),
    on=cols_victima, how='left'
)

# Unimos victimas (grano del hecho) con hechos (tiempo, ubicación) por id_hecho
fact_siniestros = df_victimas.merge(
    df_hechos[['id_hecho', 'id_tiempo', 'id_ubicacion']],
    on='id_hecho', how='left'
)

fact_siniestros['cantidad_victimas'] = 1
fact_siniestros['es_fallecido'] = (fact_siniestros['gravedad'].str.upper() == 'MORTAL').astype(int)

columnas_fact = [
    'id_hecho',
    'id_tiempo',
    'id_ubicacion',
    'id_victima_perfil',
    'cantidad_victimas',
    'es_fallecido'
]

fact_siniestros = fact_siniestros[columnas_fact]
fact_siniestros = fact_siniestros.dropna(subset=['id_tiempo', 'id_ubicacion', 'id_victima_perfil']).reset_index(drop=True)

print(fact_siniestros.head())
print("Filas en Fact_Siniestros:", len(fact_siniestros))

## 6. Guardado de los archivos del Data Warehouse

In [ ]:
ruta_guardado = '/content/drive/MyDrive/TPDataWarehouseSiniestros/'

fact_siniestros.to_csv(ruta_guardado + 'DW_Fact_Siniestros.csv', index=False)
dim_tiempo.to_csv(ruta_guardado + 'DW_Dim_Tiempo.csv', index=False)
dim_ubicacion.to_csv(ruta_guardado + 'DW_Dim_Ubicacion.csv', index=False)
dim_victima.to_csv(ruta_guardado + 'DW_Dim_Victima.csv', index=False)

print("Archivos guardados en:", ruta_guardado)